# Romanian-Transformers Model Evaluation

Use this colab to evaluate _tranformer model performance_. We currently support:

*   [**Named Entity Recognition**](https://github.com/dumitrescustefan/ronec), based on RONECv2
*   [**Part of Speech Tagging**](https://github.com/dumitrescustefan/ro-pos-tagger), based on UD's Ro-RRT dataset
*   [**Semantic Textual Simiarity**](https://github.com/dumitrescustefan/RO-STS), based on RO-STS
*   [**Emotion Detection in Tweets**](https://github.com/Alegzandra/RED-Romanian-Emotions-Dataset), based on REDv2
*   [**Perplexity**](https://github.com/dumitrescustefan/wiki-ro), based on wiki-ro, only for generative models

**How to use:**
1. Choose the model in the dropdown below.
2. Choose the task
3. Choose the number of iterations (how many times to run the same task and average over)
4. Run all cells ($\color{red}{\text{Ctrl+F9}}$); you will see the averaged results printed at the bottom, as well as saved as jsons in each task's respective folder.

** Note: Use a GPU runtime and a browser addon (like Colab Auto Reconnect) to keep this session open - some tasks, with 5 iterations, might take some hours to complete.
This is the official script used to measure model performance on
the [Romanian-Transformers repo](https://github.com/dumitrescustefan/Romanian-Transformers).


---



In [27]:
#@title Evaluation parameters

model = "dumitrescustefan/bert-base-romanian-cased-v1" # @param ["dumitrescustefan/bert-base-romanian-cased-v1","bert-base-multilingual-cased","google/mt5-base","google/rembert"] {"allow-input":true}
task = 'Named Entity Recognition' #@param ["Named Entity Recognition", "POS Tagging", "Emotion Detection in Tweets", "Semantic Textual Similarity", "CLM Perplexity"]
TRAIN_COND = "diac" #@param ["diac", "nodiac"]
iteration_no = "second" #@param ["first", "second", "third"]

iterations = '1'
print("\nWe are going to eval \033[92m{}\033[0m on the \033[92m{}\033[0m task for \033[92m{}\033[0m iteration(s).\n".format(model, task, iterations))

import torch
if not torch.cuda.is_available():
  print(f"\033[101m*** Please use a GPU-enabled colab! ***\033[0m")
else:
  print(f"\nRunning on a \033[92m{torch.cuda.get_device_name(0)}\033[0m with \033[92m{torch.cuda.get_device_properties(0).total_memory/1024/1024/1024:.0f}GB\033[0m RAM.")


We are going to eval dumitrescustefan/bert-base-romanian-cased-v1 on the Named Entity Recognition task for 1 iteration(s).


Running on a Tesla T4 with 15GB RAM.


##### Code section, run every cell automatically with Ctrl+F9

In [28]:
MODEL_KEY = {
    "dumitrescustefan/bert-base-romanian-cased-v1": "robert",
    "bert-base-multilingual-cased": "mbert",
    "google/mt5-base": "mt5",
    "google/rembert": "rembert"
}[model]

RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

SEED = {
    "first": 315,
    "second": 222,
    "third": 156
}[iteration_no]

print("Run name: " + RUN_NAME)
print("Seed: " + str(SEED))

Run name: robert_train_diac
Seed: 222


In [33]:
BASE     = f"/kaggle/working/{RUN_NAME}"
CKPT_DIR = f"{BASE}/checkpoints"
RES_DIR  = f"{BASE}/results"
PRED_DIR = f"{BASE}/predictions"

for d in [CKPT_DIR, RES_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

In [30]:
# configs
batch_size, accumulate_grad_batches = 8, 1
if "-large" in model or "-medium" in model:
  batch_size = 1
  accumulate_grad_batches = 8

In [31]:
diac_train_file='/kaggle/input/datasets/tomaalexandra06/ronec-diac/train.json'
diac_validation_file='/kaggle/input/datasets/tomaalexandra06/ronec-diac/valid.json'
diac_test_file='/kaggle/input/datasets/tomaalexandra06/ronec-diac/test.json'

In [35]:
def eval_ner(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/Alexandra06T/RoDi-Utils.git
  !pip3 install -r RoDi-Utils/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd RoDi-Utils && python evaluate.py --batch_size={batch_size} --accumulate_grad_batches={accumulate_grad_batches} --model_name {model} --train_file {diac_train_file} --validation_file {diac_validation_file} --test_file {diac_test_file} --dirpath {CKPT_DIR} --seed {SEED}

  return None

def eval_pos(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/dumitrescustefan/ro-pos-tagger.git
  !pip3 install -r ro-pos-tagger/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd ro-pos-tagger/model && python evaluate_pos_tagger.py --experiment_iterations $iterations --model_name $model --batch_size=$batch_size

  return None

def eval_sts(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/dumitrescustefan/RO-STS.git
  !pip3 install -r RO-STS/baseline-models/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd RO-STS/baseline-models && python transformer_model.py --experiment_iterations $iterations --model_name $model --batch_size=$batch_size --accumulate_grad_batches=$accumulate_grad_batches

  return None

In [ ]:
# run
if task == "Named Entity Recognition":
  eval_ner(model, iterations, batch_size, accumulate_grad_batches)
if task == "POS Tagging":
  eval_pos(model, iterations, batch_size, accumulate_grad_batches)
if task == "Semantic Textual Similarity":
  eval_sts(model, iterations, batch_size, accumulate_grad_batches)

Preparing environment ...
fatal: destination path 'RoDi-Utils' already exists and is not an empty directory.
Running task ...
Seed set to 222
Loading data...
	Dataset contains 31 BIO2 classes: ['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-NAT_REL_POL', 'I-NAT_REL_POL', 'B-EVENT', 'I-EVENT', 'B-LANGUAGE', 'I-LANGUAGE', 'B-WORK_OF_ART', 'I-WORK_OF_ART', 'B-DATETIME', 'I-DATETIME', 'B-PERIOD', 'I-PERIOD', 'B-MONEY', 'I-MONEY', 'B-QUANTITY', 'I-QUANTITY', 'B-NUMERIC', 'I-NUMERIC', 'B-ORDINAL', 'I-ORDINAL', 'B-FACILITY', 'I-FACILITY'].
	There are 16 classes: ['O', 'PERSON', 'ORG', 'GPE', 'LOC', 'NAT_REL_POL', 'EVENT', 'LANGUAGE', 'WORK_OF_ART', 'DATETIME', 'PERIOD', 'MONEY', 'QUANTITY', 'NUMERIC', 'ORDINAL', 'FACILITY']

2026-07-26 14:13:21 httpx INFO: HTTP Request: HEAD https://huggingface.co/dumitrescustefan/bert-base-romanian-cased-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:13:21 httpx INFO: HTTP Request: HEAD http

## Save checkpoint locally

In [19]:
chk_file = !ls {CKPT_DIR}
chk_path = CKPT_DIR + "/" + chk_file[0]
chk_path

In [22]:
zip_name = RUN_NAME + "_"+ iteration_no + ".zip"
zip_path = "/kaggle/working/" + zip_name
!zip -r {zip_path} {chk_path}

updating: kaggle/working/mbert_train_diac/checkpoints/epoch=7.ckpt (deflated 35%)


In [25]:
from IPython.display import FileLink
%cd /kaggle/working
FileLink(zip_name)

/kaggle/working


/kaggle/working/mbert_diac_third.zip